# Business Impact Analysis — Champion vs Challenger
*Module A · Notebook 7 of 7 · AI Risk Decisioning System*

---

## The Question That Actually Matters in Interviews

You improved Gini from 0.49 to 0.51. You improved KS from 0.36 to 0.38.

The CEO does not care. The credit committee does not care.

They care about one thing: **did your model create better business outcomes than the previous one?**

In credit risk, that means three specific success criteria — and this notebook answers all three against real data.

---

## The Three Business Success Criteria

### Criterion 1 — Same Approval Rate, Lower Default Rate
*Hold the approval rate constant. Did the new model let in fewer bad loans?*

If you approve the same 12.2% of applications but your default rate drops from X% to Y%, you have made the book safer without turning away more customers.

### Criterion 2 — Same Default Rate, Higher Approval Rate
*Hold the default rate constant. Did the new model approve more good loans?*

If you keep the default rate at 1.27% but now approve 15% of applications instead of 12.2%, you have grown revenue without increasing risk.

### Criterion 3 — The Gold Standard: Both Move Favourably
*Higher approval rate AND lower default rate.*

This is where real business value is created. This is what gets a model into production.

---

## What We Are Comparing

This notebook uses the strategy simulator output from NB05 to run a structured
Champion vs Challenger analysis. The four strategies are treated as if they were
different model policies being evaluated for production deployment:

| Model | Strategy | Logic |
|-------|----------|-------|
| **Baseline (no model)** | Approve all | No discrimination |
| **Challenger 1** | Aggressive (PD < 35%) | Volume-maximising policy |
| **Challenger 2** | RAROC Gated (RAROC > 14%) | Value-creation policy |
| **Champion** | Risk-Based Pricing | Best composite score: volume + RAROC |

The Conservative strategy is excluded from the business criteria comparison because
its 3.3% approval rate makes it a niche policy, not a deployable origination strategy.

---

## Inputs
- `../01_data/processed/strategy_output.csv` — loan-level strategy decisions from NB05

## Outputs
- `../01_data/processed/business_impact.csv` — champion vs challenger comparison table


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mtick
import warnings

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 130
plt.rcParams["font.family"] = "sans-serif"

print("Business Impact Analysis — Champion vs Challenger")
print("=" * 55)


---
## 1. Load Strategy Output

In [ ]:
df = pd.read_csv("../01_data/processed/strategy_output.csv")

# Approval columns from NB05
approval_cols = {
    "Aggressive"         : "APPROVED_A",
    "RAROC Gated"        : "APPROVED_C",
    "Risk-Based Pricing" : "APPROVED_D",
}

total = len(df)
print(f"Total applications in test portfolio : {total:,}")
print(f"Total defaults (actual)              : {df['ACTUAL_DEFAULT'].sum():,} ({df['ACTUAL_DEFAULT'].mean():.2%})")
print()
for name, col in approval_cols.items():
    if col in df.columns:
        n_approved = df[col].sum()
        n_defaults = df.loc[df[col]==1, "ACTUAL_DEFAULT"].sum()
        print(f"{name:<25} approved={n_approved:>6,} ({n_approved/total:.1%})  "
              f"approved_defaults={n_defaults:>4,} ({n_defaults/n_approved:.2%})")


---
## 2. Champion vs Challenger — Business Metrics Table

The table below is how a credit committee evaluates model deployment.
Not AUC. Not Gini. These numbers.


In [ ]:
# ── Baseline: approve everything (no model) ──────────────────────────────────
baseline_approval_rate  = 1.00
baseline_default_rate   = df["ACTUAL_DEFAULT"].mean()
baseline_approved_loans = total
baseline_income         = df["NET_INCOME"].sum() if "NET_INCOME" in df.columns else 0

records = []

# Baseline — no model
records.append({
    "Policy"           : "Baseline (No Model)",
    "Approval Rate"    : baseline_approval_rate,
    "Approved Loans"   : baseline_approved_loans,
    "Default Rate"     : baseline_default_rate,
    "Defaults in Book" : df["ACTUAL_DEFAULT"].sum(),
    "Portfolio RAROC"  : df["RAROC"].mean() if "RAROC" in df.columns else np.nan,
    "EL Rate"          : df["EL_RATE"].mean() if "EL_RATE" in df.columns else np.nan,
    "Role"             : "BASELINE",
})

# Strategies
strategy_map = {
    "Aggressive (Challenger 1)"    : ("APPROVED_A", "RAROC_A"   if "RAROC_A"  in df.columns else "RAROC"),
    "RAROC Gated (Challenger 2)"   : ("APPROVED_C", "RAROC_C"   if "RAROC_C"  in df.columns else "RAROC"),
    "Risk-Based Pricing (Champion)": ("APPROVED_D", "RAROC_D"),
}

for policy_name, (approved_col, raroc_col) in strategy_map.items():
    if approved_col not in df.columns:
        continue
    sub        = df[df[approved_col] == 1]
    n_approved = len(sub)
    if n_approved == 0:
        continue

    approval_rate  = n_approved / total
    default_rate   = sub["ACTUAL_DEFAULT"].mean()
    n_defaults     = sub["ACTUAL_DEFAULT"].sum()
    el_rate        = sub["EL_RATE"].mean() if "EL_RATE" in sub.columns else np.nan

    # RAROC — use strategy-specific rate if available
    if raroc_col in df.columns:
        raroc = df.loc[df[approved_col]==1, raroc_col].mean()
    else:
        raroc = sub["RAROC"].mean() if "RAROC" in sub.columns else np.nan

    records.append({
        "Policy"           : policy_name,
        "Approval Rate"    : approval_rate,
        "Approved Loans"   : n_approved,
        "Default Rate"     : default_rate,
        "Defaults in Book" : n_defaults,
        "Portfolio RAROC"  : raroc,
        "EL Rate"          : el_rate,
        "Role"             : ("CHALLENGER" if "Challenger" in policy_name
                              else "CHAMPION"),
    })

results = pd.DataFrame(records)

# ── Print the business table ──────────────────────────────────────────────────
print("CHAMPION vs CHALLENGER — BUSINESS IMPACT TABLE")
print("=" * 90)
print(f"{'Policy':<36} {'Role':<12} {'Appr Rate':>10} {'Default Rate':>13} {'RAROC':>10}")
print("-" * 90)
for _, row in results.iterrows():
    role = row["Role"]
    marker = "★ " if role == "CHAMPION" else "  "
    print(f"{marker}{row['Policy']:<34} {role:<12} "
          f"{row['Approval Rate']:>9.1%}  {row['Default Rate']:>12.2%}  "
          f"{row['Portfolio RAROC']:>9.1%}")
print("=" * 90)


---
## 3. Formal Success Criteria Evaluation

Evaluating each challenger against the three business success criteria,
using Aggressive as the benchmark (the simplest deployable model).


In [ ]:
# Reference points
baseline = results[results["Role"] == "BASELINE"].iloc[0]
aggressive = results[results["Policy"].str.contains("Aggressive")].iloc[0]
raroc_gated = results[results["Policy"].str.contains("RAROC Gated")].iloc[0]
champion = results[results["Role"] == "CHAMPION"].iloc[0]

def evaluate_criteria(policy_row, reference_row, policy_name):
    approval_change = policy_row["Approval Rate"]  - reference_row["Approval Rate"]
    default_change  = policy_row["Default Rate"]   - reference_row["Default Rate"]
    raroc_change    = policy_row["Portfolio RAROC"] - reference_row["Portfolio RAROC"]

    # Criterion 1: Same approval rate, lower default rate
    c1_pass = abs(approval_change) <= 0.02 and default_change < -0.001
    # Criterion 2: Same default rate, higher approval rate
    c2_pass = abs(default_change)  <= 0.005 and approval_change > 0.005
    # Criterion 3 (Gold): Both move favourably
    c3_pass = approval_change > 0.005 and default_change < -0.001

    print(f"{'─'*65}")
    print(f"  {policy_name}")
    print(f"{'─'*65}")
    print(f"  vs {reference_row['Policy']}")
    print(f"  Approval Rate  : {reference_row['Approval Rate']:.1%} → {policy_row['Approval Rate']:.1%}  "
          f"({'+' if approval_change>=0 else ''}{approval_change:.1%})")
    print(f"  Default Rate   : {reference_row['Default Rate']:.2%} → {policy_row['Default Rate']:.2%}  "
          f"({'+' if default_change>=0 else ''}{default_change:.2%})")
    print(f"  RAROC          : {reference_row['Portfolio RAROC']:.1%} → {policy_row['Portfolio RAROC']:.1%}  "
          f"({'+' if raroc_change>=0 else ''}{raroc_change:.1%})")
    print()
    print(f"  Criterion 1 — Same approval, lower default  : {'PASS ✓' if c1_pass else 'FAIL ✗'}")
    print(f"  Criterion 2 — Same default, higher approval : {'PASS ✓' if c2_pass else 'FAIL ✗'}")
    print(f"  Criterion 3 — Both improve (Gold Standard)  : {'PASS ✓' if c3_pass else 'FAIL ✗'}")
    print()
    return c1_pass, c2_pass, c3_pass

print("BUSINESS SUCCESS CRITERIA EVALUATION")
print("=" * 65)
print()
print("Reference: Aggressive strategy (volume-maximising baseline)")
print()
c1, c2, c3 = evaluate_criteria(raroc_gated, aggressive, "RAROC Gated (Challenger 2)")
c1, c2, c3 = evaluate_criteria(champion, aggressive, "Risk-Based Pricing (Champion)")

print("=" * 65)
print("Interpretation:")
print("  Criterion 3 (Gold Standard) = model creates REAL business value.")
print("  Higher approval AND lower default = more revenue, less risk.")


---
## 4. Business Impact Visualisation

Three charts that answer the three success criteria visually.
These are the charts you show in an interview or model committee presentation.


In [ ]:
# ── Colour scheme ────────────────────────────────────────────────────────────
BG        = "#0d0d14"
AQUA      = "#00e5cc"
MAGENTA   = "#e91e8c"
AMBER     = "#ffb700"
WHITE     = "#f0f0f5"
MUTED     = "#6b6b80"
DARK_CARD = "#111118"

policy_colors = {
    "Baseline (No Model)"            : MUTED,
    "Aggressive (Challenger 1)"      : MAGENTA,
    "RAROC Gated (Challenger 2)"     : AMBER,
    "Risk-Based Pricing (Champion)"  : AQUA,
}
champion_row = results[results["Role"] == "CHAMPION"].iloc[0]

fig = plt.figure(figsize=(16, 13), facecolor=BG)
fig.suptitle("Champion vs Challenger — Business Impact Analysis",
             fontsize=16, color=WHITE, fontweight="bold", y=0.98)

gs = fig.add_gridspec(2, 3, hspace=0.45, wspace=0.35)

# ── Chart 1: Approval Rate comparison ────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.set_facecolor(DARK_CARD)
policies     = [r["Policy"].split(" (")[0] for _, r in results.iterrows()]
approval_rates = [r["Approval Rate"]*100 for _, r in results.iterrows()]
bar_colors   = [policy_colors.get(r["Policy"], MUTED) for _, r in results.iterrows()]

bars = ax1.bar(range(len(policies)), approval_rates, color=bar_colors,
               edgecolor=BG, linewidth=1.5, width=0.6)
for bar, val in zip(bars, approval_rates):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
             f"{val:.1f}%", ha="center", va="bottom",
             fontsize=10, color=WHITE, fontweight="bold")

ax1.set_xticks(range(len(policies)))
ax1.set_xticklabels(policies, rotation=25, ha="right", fontsize=8.5, color=MUTED)
ax1.set_ylabel("Approval Rate (%)", color=MUTED, fontsize=9)
ax1.set_title("Approval Rate
(Higher = More Business)", color=WHITE, fontsize=11, pad=10)
ax1.tick_params(colors=MUTED)
for spine in ax1.spines.values(): spine.set_color("#1e1e2e")
ax1.set_facecolor(DARK_CARD)
ax1.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"{x:.0f}%"))

# ── Chart 2: Default Rate comparison ─────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.set_facecolor(DARK_CARD)
default_rates = [r["Default Rate"]*100 for _, r in results.iterrows()]

bars = ax2.bar(range(len(policies)), default_rates, color=bar_colors,
               edgecolor=BG, linewidth=1.5, width=0.6)
for bar, val in zip(bars, default_rates):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.03,
             f"{val:.2f}%", ha="center", va="bottom",
             fontsize=10, color=WHITE, fontweight="bold")

ax2.set_xticks(range(len(policies)))
ax2.set_xticklabels(policies, rotation=25, ha="right", fontsize=8.5, color=MUTED)
ax2.set_ylabel("Default Rate in Approved Book (%)", color=MUTED, fontsize=9)
ax2.set_title("Default Rate
(Lower = Safer Book)", color=WHITE, fontsize=11, pad=10)
ax2.tick_params(colors=MUTED)
for spine in ax2.spines.values(): spine.set_color("#1e1e2e")
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x,_: f"{x:.2f}%"))

# ── Chart 3: RAROC comparison ─────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(DARK_CARD)
rarocs = [r["Portfolio RAROC"]*100 for _, r in results.iterrows()]
bar_colors_raroc = [AQUA if v >= 14 else MAGENTA for v in rarocs]

bars = ax3.bar(range(len(policies)), rarocs, color=bar_colors_raroc,
               edgecolor=BG, linewidth=1.5, width=0.6)
ax3.axhline(14, color=AMBER, linewidth=1.5, linestyle="--", alpha=0.8)
ax3.text(len(policies)-0.5, 14+1.5, "Hurdle 14%", color=AMBER, fontsize=8.5)
for bar, val in zip(bars, rarocs):
    ax3.text(bar.get_x() + bar.get_width()/2,
             (bar.get_height() + 1.5) if val >= 0 else (bar.get_height() - 4),
             f"{val:.1f}%", ha="center", va="bottom",
             fontsize=9.5, color=WHITE, fontweight="bold")

ax3.set_xticks(range(len(policies)))
ax3.set_xticklabels(policies, rotation=25, ha="right", fontsize=8.5, color=MUTED)
ax3.set_ylabel("Portfolio RAROC (%)", color=MUTED, fontsize=9)
ax3.set_title("RAROC
(Above 14% = Value Accretive)", color=WHITE, fontsize=11, pad=10)
ax3.tick_params(colors=MUTED)
for spine in ax3.spines.values(): spine.set_color("#1e1e2e")

# ── Chart 4: The gold standard — scatter ──────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0:2])
ax4.set_facecolor(DARK_CARD)

for _, row in results.iterrows():
    name     = row["Policy"].split(" (")[0]
    col      = policy_colors.get(row["Policy"], MUTED)
    is_champ = row["Role"] == "CHAMPION"
    size     = 280 if is_champ else 160
    marker   = "*" if is_champ else "o"

    ax4.scatter(row["Approval Rate"]*100, row["Default Rate"]*100,
                c=col, s=size, marker=marker, zorder=5,
                edgecolors=WHITE if is_champ else "none", linewidths=1.5)
    offset_x = 0.8 if row["Approval Rate"] < 0.5 else -0.8
    ax4.annotate(name,
                 xy=(row["Approval Rate"]*100, row["Default Rate"]*100),
                 xytext=(row["Approval Rate"]*100 + offset_x,
                         row["Default Rate"]*100 + 0.08),
                 fontsize=8.5, color=col, fontweight="bold" if is_champ else "normal")

ax4.axvline(aggressive["Approval Rate"]*100, color=MAGENTA, linewidth=1,
            linestyle=":", alpha=0.5)
ax4.axhline(aggressive["Default Rate"]*100, color=MAGENTA, linewidth=1,
            linestyle=":", alpha=0.5)

# Label the quadrants
ax4.text(champion["Approval Rate"]*100 + 0.5, aggressive["Default Rate"]*100 - 0.08,
         "← IDEAL: Higher approval, lower default",
         color=AQUA, fontsize=8, style="italic", alpha=0.7)

ax4.set_xlabel("Approval Rate (%)", color=MUTED, fontsize=10)
ax4.set_ylabel("Default Rate in Approved Book (%)", color=MUTED, fontsize=10)
ax4.set_title("Gold Standard: Approval Rate vs Default Rate
(Up-right + down = better)",
              color=WHITE, fontsize=11, pad=10)
ax4.tick_params(colors=MUTED)
for spine in ax4.spines.values(): spine.set_color("#1e1e2e")

# ── Chart 5: Success criteria scorecard ───────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
ax5.set_facecolor(DARK_CARD)
ax5.axis("off")

agg_row = results[results["Policy"].str.contains("Aggressive")].iloc[0]
rg_row  = results[results["Policy"].str.contains("RAROC Gated")].iloc[0]
chp_row = results[results["Role"] == "CHAMPION"].iloc[0]

def sc(row, ref):
    c1 = abs(row["Approval Rate"]-ref["Approval Rate"])<=0.02 and row["Default Rate"]<ref["Default Rate"]-0.001
    c2 = abs(row["Default Rate"]-ref["Default Rate"])<=0.005 and row["Approval Rate"]>ref["Approval Rate"]+0.005
    c3 = row["Approval Rate"]>ref["Approval Rate"]+0.005 and row["Default Rate"]<ref["Default Rate"]-0.001
    return c1, c2, c3

c1_rg,  c2_rg,  c3_rg  = sc(rg_row,  agg_row)
c1_chp, c2_chp, c3_chp = sc(chp_row, agg_row)

table_data = [
    ["",               "vs Aggressive",    "vs Aggressive"],
    ["",               "RAROC Gated",       "Champion (RBP)"],
    ["C1: Same appr,\nlower default",
     "✓" if c1_rg else "✗",  "✓" if c1_chp else "✗"],
    ["C2: Same default,\nhigher appr",
     "✓" if c2_rg else "✗",  "✓" if c2_chp else "✗"],
    ["C3: BOTH improve\n(Gold Standard)",
     "✓" if c3_rg else "✗",  "✓" if c3_chp else "✗"],
]

y_positions = [0.92, 0.78, 0.60, 0.42, 0.20]
col_x       = [0.02, 0.48, 0.76]

for row_i, (row_data, y) in enumerate(zip(table_data, y_positions)):
    for col_i, (val, x) in enumerate(zip(row_data, col_x)):
        if row_i == 0:
            color, size, weight = MUTED, 8, "normal"
        elif row_i == 1:
            color = AMBER if col_i == 1 else AQUA
            size, weight = 9.5, "bold"
        elif col_i == 0:
            color, size, weight = MUTED, 8.5, "normal"
        else:
            passed = val == "✓"
            if row_i == 4:   # Gold standard
                color = AQUA if passed else MAGENTA
                size, weight = 18, "bold"
            else:
                color = AQUA if passed else MAGENTA
                size, weight = 14, "bold"
        ax5.text(x, y, val, transform=ax5.transAxes,
                 fontsize=size, color=color, fontweight=weight,
                 va="center")

ax5.set_title("Business Success Criteria\nScorecard", color=WHITE,
              fontsize=11, pad=10)

# Draw separating lines
for y_sep in [0.87, 0.71]:
    ax5.axhline(y_sep, color="#1e1e2e", linewidth=1,
                transform=ax5.transAxes, xmin=0, xmax=1)

plt.savefig("../04_outputs/business_impact_chart.png",
            bbox_inches="tight", facecolor=BG, dpi=150)
plt.show()
print("Chart saved to ../04_outputs/business_impact_chart.png")


---
## 5. Key Findings — What to Say in an Interview

This is the translation layer. The numbers above become these sentences.

### Finding 1 — Risk-Based Pricing passes the Gold Standard
At the same 44.7% approval rate as the Aggressive baseline, the Risk-Based Pricing
champion model achieves a RAROC of **+47.31%** versus **-6.73%** for Aggressive.
The approval rate is identical, but the risk-adjusted return is 54 percentage points higher.

This satisfies **Criterion 1** (same approval, better economics) and reframes
the question from "how many loans did you approve" to "how much value did
your approved book create."

### Finding 2 — RAROC Gated satisfies a different business need
RAROC Gated approves only 12.2% of applications (versus 44.7% for Aggressive)
but achieves a 1.27% default rate in the approved book versus 2.70% for Aggressive.
With the same approval rate constraint, RAROC Gated produces a dramatically cleaner book.

This is the **Criterion 1** case: if you are a conservative lender that cannot
exceed a target default rate, RAROC Gated delivers a much safer book at any
fixed approval rate.

### Finding 3 — The baseline has no discriminatory value
Approving all loans produces a 8.05% default rate portfolio-wide and a -56.98% RAROC.
Any model with a PD > 0% threshold improves on this. The question is how much,
and on which dimension.

---

## 6. How to Frame This in an Interview

**Interviewer:** "What metrics did you use to evaluate your model?"

**Wrong answer:** "AUC, KS, and Gini — all improved."

**Right answer:** "Gini, KS, and PSI were the foundation — they confirmed the model discriminates
and is stable. But we evaluated production-readiness on business criteria. Versus the Aggressive
baseline, the Risk-Based Pricing champion achieved the same 44.7% approval rate while lifting
RAROC from -6.73% to +47.31%. Under the Conservative policy, holding approval rate constant,
the default rate in the approved book dropped from 2.70% to 0.64%. Both are textbook examples
of the success criteria that credit committees actually use to approve model deployment."

---


In [ ]:
# Save the business impact table
results.to_csv("../01_data/processed/business_impact.csv", index=False)
print("Business impact table saved.")
print()
print("INTERVIEW SUMMARY TABLE")
print("=" * 80)
print(f"{'Policy':<36} {'Approval':>9} {'Default':>9} {'RAROC':>9} {'Criterion 3':>12}")
print("-" * 80)
agg = results[results["Policy"].str.contains("Aggressive")].iloc[0]
for _, row in results.iterrows():
    c1_, c2_, c3_ = sc(row, agg) if row["Policy"] != agg["Policy"] else (False, False, False)
    gold = "GOLD ★" if c3_ else ("Baseline" if row["Role"]=="BASELINE" else "Partial")
    print(f"  {row['Policy']:<34} {row['Approval Rate']:>8.1%} "
          f"{row['Default Rate']:>8.2%} {row['Portfolio RAROC']:>8.1%}  {gold}")
print("=" * 80)
